<a href="https://colab.research.google.com/github/Arashi283/AIRepoOne/blob/main/cleaned_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.7 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model
from datasets import load_dataset

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# 4-bit quantization config (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

# Load model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

# LoRA config
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

# Load dataset
dataset = load_dataset("text", data_files={"train": "train.txt"})

def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = dataset.map(tokenize_function, batched=True)

training_args = TrainingArguments(
    output_dir="./story-qlora",
    per_device_train_batch_size=4,
    num_train_epochs=1,
    max_steps=500,
    logging_steps=20,
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
)

trainer.train()

model.save_pretrained("story-qlora")
tokenizer.save_pretrained("story-qlora")

print("QLoRA training complete.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/730 [00:00<?, ? examples/s]

Step,Training Loss
20,4.158768
40,2.317227
60,1.127581
80,1.038110
100,0.667200
120,0.777063
140,0.845876
160,1.011457
180,0.797882
200,0.935384


QLoRA training complete.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_PATH = "story-qlora"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH)

prompt = """<|user|>
Write a complete short story with a clear beginning, middle, and ending about a lonely traveler in a magical forest. End the story conclusively.
<|assistant|>
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=500,
    temperature=0.8,
    do_sample=True
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

<|user|>
Write a complete short story with a clear beginning, middle, and ending about a lonely traveler in a magical forest. End the story conclusively.
<|assistant|>
Title: The Forest of Dreams

The lonely traveler, Amelia, stumbled upon a vast, magical forest that seemed to stretch out endlessly. The trees were tall and sturdy, their bark smooth and dark, while the leaves rustled gently in the gentle breeze. The air was sweet with the fragrance of wildflowers, and Amelia could hear the mournful cries of a bird perched on a branch above her head.

Amelia had been wandering in search of her true home for years, ever since she lost her parents in a tragic accident. The forest had been a source of solace and a sanctuary for her, a place where she could escape the harsh realities of the world and focus on the beauty and magic around her. But as she had ventured deeper into the forest, Amelia had stumbled upon a clearing that seemed to be at the center of everything.

The clearing was sur

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16
)

prompt = """<|user|>
Write a complete short story with a clear beginning, middle, and ending about a lonely traveler in a magical forest. End the story conclusively.
<|assistant|>
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=400,
    temperature=0.8,
    do_sample=True
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


KeyboardInterrupt: 